In [1]:
import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from scipy.stats import skew, kurtosis
from scipy.signal import find_peaks

import lightgbm as lgb

from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight

In [2]:
TRAIN_DIR = "/kaggle/input/datasets/cheukhongtang/ass3data/train/train"
TEST_DIR = "/kaggle/input/datasets/cheukhongtang/ass3data/test/test"
SUB_PATH = "/kaggle/input/datasets/cheukhongtang/ass3data/sample_submission.csv"

train_files = sorted(glob.glob(os.path.join(TRAIN_DIR, "User_*", "*.csv")))
test_files = sorted(glob.glob(os.path.join(TEST_DIR, "User_*", "*.csv")))

print("Train files:", len(train_files))
print("Test files:", len(test_files))
print("Sample submission exists:", os.path.exists(SUB_PATH))


Train files: 11020
Test files: 6849
Sample submission exists: True


In [3]:
def get_user_from_path(path):
    parts = path.replace("\\", "/").split("/")
    for p in parts:
        if p.startswith("User_"):
            return p
    return "Unknown"


def safe_skew(x):
    if np.std(x) < 1e-12:
        return 0.0
    return skew(x, nan_policy="omit")


def safe_kurtosis(x):
    if np.std(x) < 1e-12:
        return 0.0
    return kurtosis(x, nan_policy="omit")


def add_basic_signal_features(df):
    df = df.copy()

    df["acc_mag"] = np.sqrt(df["mean_x"]**2 + df["mean_y"]**2 + df["mean_z"]**2)
    df["std_mag"] = np.sqrt(df["std_x"]**2 + df["std_y"]**2 + df["std_z"]**2)

    df["xy_mag"] = np.sqrt(df["mean_x"]**2 + df["mean_y"]**2)
    df["xz_mag"] = np.sqrt(df["mean_x"]**2 + df["mean_z"]**2)
    df["yz_mag"] = np.sqrt(df["mean_y"]**2 + df["mean_z"]**2)

    df["std_xy_mag"] = np.sqrt(df["std_x"]**2 + df["std_y"]**2)
    df["std_xz_mag"] = np.sqrt(df["std_x"]**2 + df["std_z"]**2)
    df["std_yz_mag"] = np.sqrt(df["std_y"]**2 + df["std_z"]**2)

    df["mean_sum"] = df["mean_x"] + df["mean_y"] + df["mean_z"]
    df["std_sum"] = df["std_x"] + df["std_y"] + df["std_z"]

    return df

In [4]:
def add_stat_features(row, values, prefix):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        values = np.array([0.0])

    q05, q10, q25, q75, q90, q95 = np.quantile(
        values, [0.05, 0.10, 0.25, 0.75, 0.90, 0.95]
    )

    mean_val = np.mean(values)

    stats = {
        "mean": mean_val,
        "std": np.std(values),
        "min": np.min(values),
        "max": np.max(values),
        "range": np.max(values) - np.min(values),
        "median": np.median(values),
        "q05": q05,
        "q10": q10,
        "q25": q25,
        "q75": q75,
        "q90": q90,
        "q95": q95,
        "iqr": q75 - q25,
        "energy": np.mean(values ** 2),
        "rms": np.sqrt(np.mean(values ** 2)),
        "mad": np.mean(np.abs(values - mean_val)),
        "skew": safe_skew(values),
        "kurtosis": safe_kurtosis(values),
    }

    for name, value in stats.items():
        row[f"{prefix}_{name}"] = value


def add_diff_features(row, values, prefix):
    values = np.asarray(values, dtype=float)

    if len(values) < 2:
        diffs = np.array([0.0])
    else:
        diffs = np.diff(values)

    row[f"{prefix}_diff_mean"] = np.mean(diffs)
    row[f"{prefix}_diff_std"] = np.std(diffs)
    row[f"{prefix}_diff_abs_mean"] = np.mean(np.abs(diffs))
    row[f"{prefix}_diff_abs_max"] = np.max(np.abs(diffs))
    row[f"{prefix}_diff_energy"] = np.mean(diffs ** 2)

    row[f"{prefix}_first_60_mean"] = np.mean(values[:60])
    row[f"{prefix}_last_60_mean"] = np.mean(values[-60:])
    row[f"{prefix}_last_minus_first_60"] = np.mean(values[-60:]) - np.mean(values[:60])

In [5]:
def add_fft_features(row, values, prefix, sample_rate=1.0):
    values = np.asarray(values, dtype=float)
    values = values - np.mean(values)

    if len(values) < 4 or np.std(values) < 1e-12:
        row[f"{prefix}_fft_dom_freq"] = 0.0
        row[f"{prefix}_fft_dom_power"] = 0.0
        row[f"{prefix}_fft_total_power"] = 0.0
        row[f"{prefix}_fft_spectral_entropy"] = 0.0
        row[f"{prefix}_fft_low_power"] = 0.0
        row[f"{prefix}_fft_high_power"] = 0.0
        return

    fft_vals = np.fft.rfft(values)
    power = np.abs(fft_vals) ** 2
    freqs = np.fft.rfftfreq(len(values), d=1.0 / sample_rate)

    power_no_dc = power[1:]
    freqs_no_dc = freqs[1:]

    if len(power_no_dc) == 0 or power_no_dc.sum() <= 0:
        row[f"{prefix}_fft_dom_freq"] = 0.0
        row[f"{prefix}_fft_dom_power"] = 0.0
        row[f"{prefix}_fft_total_power"] = 0.0
        row[f"{prefix}_fft_spectral_entropy"] = 0.0
        row[f"{prefix}_fft_low_power"] = 0.0
        row[f"{prefix}_fft_high_power"] = 0.0
        return

    dom_idx = np.argmax(power_no_dc)
    total_power = np.sum(power_no_dc)
    prob = power_no_dc / total_power

    low_mask = freqs_no_dc <= 0.10
    high_mask = freqs_no_dc > 0.10

    row[f"{prefix}_fft_dom_freq"] = freqs_no_dc[dom_idx]
    row[f"{prefix}_fft_dom_power"] = power_no_dc[dom_idx]
    row[f"{prefix}_fft_total_power"] = total_power
    row[f"{prefix}_fft_spectral_entropy"] = -np.sum(prob * np.log(prob + 1e-12))
    row[f"{prefix}_fft_low_power"] = np.sum(power_no_dc[low_mask])
    row[f"{prefix}_fft_high_power"] = np.sum(power_no_dc[high_mask])


def add_peak_features(row, values, prefix):
    values = np.asarray(values, dtype=float)

    if len(values) < 3 or np.std(values) < 1e-12:
        row[f"{prefix}_num_peaks"] = 0
        row[f"{prefix}_peak_rate"] = 0.0
        row[f"{prefix}_mean_peak_height"] = 0.0
        row[f"{prefix}_max_peak_height"] = 0.0
        return

    height_threshold = np.mean(values) + 0.5 * np.std(values)
    peaks, properties = find_peaks(values, height=height_threshold)

    row[f"{prefix}_num_peaks"] = len(peaks)
    row[f"{prefix}_peak_rate"] = len(peaks) / len(values)

    if len(peaks) > 0:
        heights = properties["peak_heights"]
        row[f"{prefix}_mean_peak_height"] = np.mean(heights)
        row[f"{prefix}_max_peak_height"] = np.max(heights)
    else:
        row[f"{prefix}_mean_peak_height"] = 0.0
        row[f"{prefix}_max_peak_height"] = 0.0

In [6]:
def add_window_features(row, df, cols, n_windows=10):
    n = len(df)
    window_size = n // n_windows

    for w in range(n_windows):
        start = w * window_size
        end = (w + 1) * window_size if w < n_windows - 1 else n
        window = df.iloc[start:end]

        for col in cols:
            values = window[col].values
            row[f"win{w}_{col}_mean"] = np.mean(values)
            row[f"win{w}_{col}_std"] = np.std(values)
            row[f"win{w}_{col}_min"] = np.min(values)
            row[f"win{w}_{col}_max"] = np.max(values)
            row[f"win{w}_{col}_energy"] = np.mean(values ** 2)

In [7]:
def summarize_file(path, is_train=True):
    df = pd.read_csv(path)
    df = df.sort_values("index").reset_index(drop=True)
    df = df.interpolate().ffill().bfill()
    df = add_basic_signal_features(df)

    row = {}
    row["path"] = path
    row["user"] = get_user_from_path(path)
    row["filename"] = os.path.basename(path)

    if "file_id" in df.columns:
        row["file_id"] = int(df["file_id"].iloc[0])
    else:
        row["file_id"] = int(os.path.splitext(os.path.basename(path))[0])

    if is_train:
        row["label"] = int(df["label"].iloc[0])

    base_cols = [
        "mean_x", "mean_y", "mean_z",
        "std_x", "std_y", "std_z",
        "acc_mag", "std_mag",
        "xy_mag", "xz_mag", "yz_mag",
        "std_xy_mag", "std_xz_mag", "std_yz_mag",
        "mean_sum", "std_sum"
    ]

    for col in base_cols:
        values = df[col].values
        add_stat_features(row, values, col)
        add_diff_features(row, values, col)

    for col in ["acc_mag", "std_mag", "std_x", "std_y", "std_z"]:
        values = df[col].values
        add_fft_features(row, values, col)
        add_peak_features(row, values, col)

    add_window_features(
        row,
        df,
        cols=["acc_mag", "std_mag", "std_x", "std_y", "std_z"],
        n_windows=10
    )

    return row


def build_feature_table(files, is_train=True):
    rows = []
    for path in tqdm(files):
        rows.append(summarize_file(path, is_train=is_train))
    return pd.DataFrame(rows)

In [8]:
train_features = build_feature_table(train_files, is_train=True)
test_features = build_feature_table(test_files, is_train=False)

print("Train features:", train_features.shape)
print("Test features:", test_features.shape)

train_features.to_csv("train_full_features_for_ablation.csv", index=False)
test_features.to_csv("test_full_features_for_ablation.csv", index=False)

  0%|          | 0/11020 [00:00<?, ?it/s]

  0%|          | 0/6849 [00:00<?, ?it/s]

Train features: (11020, 721)
Test features: (6849, 720)


In [9]:
id_cols = ["path", "user", "filename", "file_id"]
target_col = "label"

feature_cols = [
    c for c in train_features.columns
    if c not in id_cols + [target_col]
]

X = train_features[feature_cols].copy()
y = train_features[target_col].copy()
groups = train_features["user"].copy()

X = X.replace([np.inf, -np.inf], np.nan)
medians = X.median()
X = X.fillna(medians)

sample_submission = pd.read_csv(SUB_PATH)

test_features_ordered = sample_submission[["Id"]].merge(
    test_features,
    left_on="Id",
    right_on="file_id",
    how="left"
)

print("Missing test matches:", test_features_ordered["file_id"].isna().sum())

X_test_ordered = test_features_ordered[feature_cols].copy()
X_test_ordered = X_test_ordered.replace([np.inf, -np.inf], np.nan)
X_test_ordered = X_test_ordered.fillna(medians)

print("X:", X.shape)
print("X_test_ordered:", X_test_ordered.shape)

Missing test matches: 0
X: (11020, 716)
X_test_ordered: (6849, 716)


In [10]:
def columns_for_signals_and_suffixes(feature_cols, signals, suffixes):
    cols = []
    for col in feature_cols:
        for signal in signals:
            for suffix in suffixes:
                if col == f"{signal}{suffix}":
                    cols.append(col)
                    break
            else:
                continue
            break
    return cols


def columns_for_signals_and_patterns(feature_cols, signals, patterns):
    cols = []
    for col in feature_cols:
        for signal in signals:
            if col.startswith(f"{signal}_") and any(p in col for p in patterns):
                cols.append(col)
                break
    return cols


def unique_keep_order(cols):
    seen = set()
    out = []
    for c in cols:
        if c not in seen:
            out.append(c)
            seen.add(c)
    return out


raw_signals = [
    "mean_x", "mean_y", "mean_z",
    "std_x", "std_y", "std_z"
]

magnitude_signals = [
    "acc_mag", "std_mag",
    "xy_mag", "xz_mag", "yz_mag",
    "std_xy_mag", "std_xz_mag", "std_yz_mag",
    "mean_sum", "std_sum"
]

all_signals = raw_signals + magnitude_signals

basic_suffixes = [
    "_mean", "_std", "_min", "_max", "_range", "_median"
]

rich_suffixes = [
    "_q05", "_q10", "_q25", "_q75", "_q90", "_q95",
    "_iqr", "_energy", "_rms", "_mad", "_skew", "_kurtosis"
]

temporal_patterns = [
    "_diff_",
    "first_60",
    "last_60",
    "last_minus_first"
]

baseline_raw_basic = columns_for_signals_and_suffixes(
    feature_cols,
    raw_signals,
    basic_suffixes
)

magnitude_basic = columns_for_signals_and_suffixes(
    feature_cols,
    magnitude_signals,
    basic_suffixes
)

raw_rich_statistics = columns_for_signals_and_suffixes(
    feature_cols,
    raw_signals,
    rich_suffixes
)

magnitude_rich_statistics = columns_for_signals_and_suffixes(
    feature_cols,
    magnitude_signals,
    rich_suffixes
)

all_rich_statistics = raw_rich_statistics + magnitude_rich_statistics

raw_temporal_diff_trend = columns_for_signals_and_patterns(
    feature_cols,
    raw_signals,
    temporal_patterns
)

magnitude_temporal_diff_trend = columns_for_signals_and_patterns(
    feature_cols,
    magnitude_signals,
    temporal_patterns
)

all_temporal_diff_trend = raw_temporal_diff_trend + magnitude_temporal_diff_trend

window_features = [
    c for c in feature_cols
    if c.startswith("win")
]

fft_features = [
    c for c in feature_cols
    if "_fft_" in c
]

peak_features = [
    c for c in feature_cols
    if "_num_peaks" in c
    or "_peak_rate" in c
    or "_mean_peak_height" in c
    or "_max_peak_height" in c
]

feature_groups = {
    "baseline_raw_basic": baseline_raw_basic,
    "magnitude_basic": magnitude_basic,
    "raw_rich_statistics": raw_rich_statistics,
    "magnitude_rich_statistics": magnitude_rich_statistics,
    "raw_temporal_diff_trend": raw_temporal_diff_trend,
    "magnitude_temporal_diff_trend": magnitude_temporal_diff_trend,
    "window_features": window_features,
    "fft_features": fft_features,
    "peak_features": peak_features,
}

print("Feature group sizes:")
for name, cols in feature_groups.items():
    print(f"{name}: {len(cols)}")

feature_sets = {}

# Reference baseline: only simple statistics from the original x/y/z axes.
feature_sets["baseline_raw_basic_stats"] = unique_keep_order(
    baseline_raw_basic
)

# Core candidate: rich distribution + temporal-change features, but no magnitude-derived features.
feature_sets["rich_temporal_no_magnitude"] = unique_keep_order(
    baseline_raw_basic
    + raw_rich_statistics
    + raw_temporal_diff_trend
)

# This corresponds closely to the previous best additive Kaggle setup:
# raw basic + magnitude basic + rich statistics + temporal diff/trend.
feature_sets["rich_temporal_magnitude"] = unique_keep_order(
    baseline_raw_basic
    + magnitude_basic
    + all_rich_statistics
    + all_temporal_diff_trend
)

# Test whether peak/burst features help when used without windows and FFT.
feature_sets["rich_temporal_peaks"] = unique_keep_order(
    baseline_raw_basic
    + raw_rich_statistics
    + raw_temporal_diff_trend
    + peak_features
)

# Test whether local 30-second summaries help when added to the core feature set.
feature_sets["rich_temporal_windows"] = unique_keep_order(
    baseline_raw_basic
    + raw_rich_statistics
    + raw_temporal_diff_trend
    + window_features
)

# Test local time structure + burst information, but still without FFT.
feature_sets["rich_temporal_windows_peaks"] = unique_keep_order(
    baseline_raw_basic
    + raw_rich_statistics
    + raw_temporal_diff_trend
    + window_features
    + peak_features
)

# Optional check: FFT without windows, because FFT was weak in the previous additive order.
feature_sets["rich_temporal_fft"] = unique_keep_order(
    baseline_raw_basic
    + raw_rich_statistics
    + raw_temporal_diff_trend
    + fft_features
)

# A slightly larger candidate: add magnitude to the window+peak combination.
feature_sets["rich_temporal_magnitude_windows_peaks"] = unique_keep_order(
    baseline_raw_basic
    + magnitude_basic
    + all_rich_statistics
    + all_temporal_diff_trend
    + window_features
    + peak_features
)

print("\nCombination feature sets:")
for name, cols in feature_sets.items():
    print(f"{name}: {len(cols)} features")


Feature group sizes:
baseline_raw_basic: 36
magnitude_basic: 60
raw_rich_statistics: 72
magnitude_rich_statistics: 120
raw_temporal_diff_trend: 48
magnitude_temporal_diff_trend: 80
window_features: 250
fft_features: 30
peak_features: 20

Combination feature sets:
baseline_raw_basic_stats: 36 features
rich_temporal_no_magnitude: 156 features
rich_temporal_magnitude: 416 features
rich_temporal_peaks: 176 features
rich_temporal_windows: 406 features
rich_temporal_windows_peaks: 426 features
rich_temporal_fft: 186 features
rich_temporal_magnitude_windows_peaks: 686 features


In [11]:
def run_lightgbm_combination_experiment(
    experiment_name,
    selected_features,
    X,
    y,
    groups,
    X_test_ordered,
    sample_submission,
    n_splits=5,
    random_seed=42
):
    print("=" * 80)
    print("Experiment:", experiment_name)
    print("Number of features:", len(selected_features))
    print("=" * 80)

    X_exp = X[selected_features].copy()
    X_test_exp = X_test_ordered[selected_features].copy()

    X_exp = X_exp.replace([np.inf, -np.inf], np.nan)
    X_test_exp = X_test_exp.replace([np.inf, -np.inf], np.nan)

    exp_medians = X_exp.median()
    X_exp = X_exp.fillna(exp_medians)
    X_test_exp = X_test_exp.fillna(exp_medians)

    gkf = GroupKFold(n_splits=n_splits)

    oof_preds = np.zeros(len(X_exp), dtype=int)
    test_proba = np.zeros((len(X_test_exp), 6))

    fold_scores = []
    models = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_exp, y, groups), start=1):
        print(f"\n{experiment_name} - Fold {fold}")

        X_train = X_exp.iloc[train_idx]
        X_val = X_exp.iloc[val_idx]
        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        sample_weight = compute_sample_weight(
            class_weight="balanced",
            y=y_train
        )

        model = lgb.LGBMClassifier(
            objective="multiclass",
            num_class=6,
            n_estimators=2000,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=20,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=random_seed + fold,
            n_jobs=-1,
            verbose=-1
        )

        model.fit(
            X_train,
            y_train,
            sample_weight=sample_weight,
            eval_set=[(X_val, y_val)],
            eval_metric="multi_logloss",
            callbacks=[
                lgb.early_stopping(stopping_rounds=100),
                lgb.log_evaluation(period=100)
            ]
        )

        val_proba = model.predict_proba(X_val)
        val_pred = np.argmax(val_proba, axis=1)

        oof_preds[val_idx] = val_pred

        fold_score = f1_score(y_val, val_pred, average="macro")
        fold_scores.append(fold_score)

        print(f"Fold {fold} macro-F1: {fold_score:.5f}")

        test_proba += model.predict_proba(X_test_exp) / n_splits
        models.append(model)

    mean_macro_f1 = np.mean(fold_scores)
    std_macro_f1 = np.std(fold_scores)
    oof_macro_f1 = f1_score(y, oof_preds, average="macro")

    print("\nExperiment:", experiment_name)
    print("Mean macro-F1:", mean_macro_f1)
    print("Std macro-F1:", std_macro_f1)
    print("OOF macro-F1:", oof_macro_f1)
    print(classification_report(y, oof_preds, digits=4))

    test_pred = np.argmax(test_proba, axis=1)

    submission = sample_submission.copy()
    submission["Label"] = test_pred.astype(int)

    output_name = f"submission_combination_{experiment_name}.csv"
    submission.to_csv(output_name, index=False)

    print("Saved:", output_name)
    print("Prediction distribution:")
    display(submission["Label"].value_counts().sort_index())

    return {
        "experiment": experiment_name,
        "n_features": len(selected_features),
        "fold_scores": fold_scores,
        "mean_macro_f1": mean_macro_f1,
        "std_macro_f1": std_macro_f1,
        "oof_macro_f1": oof_macro_f1,
        "submission_file": output_name
    }


In [12]:
selected_experiments = [
    "baseline_raw_basic_stats",
    "rich_temporal_no_magnitude",
    "rich_temporal_magnitude",
    "rich_temporal_peaks",
    "rich_temporal_windows",
    "rich_temporal_windows_peaks",
    "rich_temporal_fft",
    "rich_temporal_magnitude_windows_peaks",
]

combination_results = []

for experiment_name in selected_experiments:
    result = run_lightgbm_combination_experiment(
        experiment_name=experiment_name,
        selected_features=feature_sets[experiment_name],
        X=X,
        y=y,
        groups=groups,
        X_test_ordered=X_test_ordered,
        sample_submission=sample_submission,
        n_splits=5,
        random_seed=42
    )

    combination_results.append(result)


Experiment: baseline_raw_basic_stats
Number of features: 36

baseline_raw_basic_stats - Fold 1
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.510918
[200]	valid_0's multi_logloss: 0.476278
[300]	valid_0's multi_logloss: 0.485711
Early stopping, best iteration is:
[219]	valid_0's multi_logloss: 0.474787
Fold 1 macro-F1: 0.62404

baseline_raw_basic_stats - Fold 2
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.457106
[200]	valid_0's multi_logloss: 0.405955
[300]	valid_0's multi_logloss: 0.404518
Early stopping, best iteration is:
[258]	valid_0's multi_logloss: 0.40172
Fold 2 macro-F1: 0.70652

baseline_raw_basic_stats - Fold 3
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.389042
[200]	valid_0's multi_logloss: 0.346821
[300]	valid_0's multi_logloss: 0.352206
Early stopping, best iteration is:
[209]	valid_0's multi_logloss: 0.346482
Fold 3 macro-F1

Label
0    2865
1    2960
2     144
3     540
4      74
5     266
Name: count, dtype: int64

Experiment: rich_temporal_no_magnitude
Number of features: 156

rich_temporal_no_magnitude - Fold 1
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.443097
[200]	valid_0's multi_logloss: 0.414925
Early stopping, best iteration is:
[194]	valid_0's multi_logloss: 0.41469
Fold 1 macro-F1: 0.65109

rich_temporal_no_magnitude - Fold 2
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.394706
[200]	valid_0's multi_logloss: 0.353336
[300]	valid_0's multi_logloss: 0.356397
Early stopping, best iteration is:
[229]	valid_0's multi_logloss: 0.351553
Fold 2 macro-F1: 0.74805

rich_temporal_no_magnitude - Fold 3
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.337624
[200]	valid_0's multi_logloss: 0.304983
[300]	valid_0's multi_logloss: 0.320209
Early stopping, best iteration is:
[209]	valid_0's multi_logloss: 0.304951
Fold 3 macro-F1: 0.74331

rich_temporal_no_mag

Label
0    2800
1    3064
2     121
3     536
4      79
5     249
Name: count, dtype: int64

Experiment: rich_temporal_magnitude
Number of features: 416

rich_temporal_magnitude - Fold 1
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.423672
[200]	valid_0's multi_logloss: 0.393533
Early stopping, best iteration is:
[181]	valid_0's multi_logloss: 0.392665
Fold 1 macro-F1: 0.66699

rich_temporal_magnitude - Fold 2
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.378416
[200]	valid_0's multi_logloss: 0.350578
Early stopping, best iteration is:
[178]	valid_0's multi_logloss: 0.349188
Fold 2 macro-F1: 0.76028

rich_temporal_magnitude - Fold 3
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.346389
[200]	valid_0's multi_logloss: 0.331941
Early stopping, best iteration is:
[165]	valid_0's multi_logloss: 0.327255
Fold 3 macro-F1: 0.73598

rich_temporal_magnitude - Fold 4
Training until validation scores don't improve for 100 rounds
[100]	valid_0's 

...

Label
0    2815
1    3077
2     101
3     512
4      90
5     254
Name: count, dtype: int64

Experiment: rich_temporal_fft
Number of features: 186

rich_temporal_fft - Fold 1
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.441494
[200]	valid_0's multi_logloss: 0.414555
Early stopping, best iteration is:
[181]	valid_0's multi_logloss: 0.41385
Fold 1 macro-F1: 0.63848

rich_temporal_fft - Fold 2
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.399469
[200]	valid_0's multi_logloss: 0.354243
[300]	valid_0's multi_logloss: 0.358616
Early stopping, best iteration is:
[224]	valid_0's multi_logloss: 0.352475
Fold 2 macro-F1: 0.74016

rich_temporal_fft - Fold 3
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.335286
[200]	valid_0's multi_logloss: 0.306105
Early stopping, best iteration is:
[180]	valid_0's multi_logloss: 0.304663
Fold 3 macro-F1: 0.74213

rich_temporal_fft - Fold 4
Training until validation scores don't improve for 100 rounds
[100]	v

Label
0    2815
1    3044
2     116
3     540
4      79
5     255
Name: count, dtype: int64

Experiment: rich_temporal_magnitude_windows_peaks
Number of features: 686

rich_temporal_magnitude_windows_peaks - Fold 1
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.407585
[200]	valid_0's multi_logloss: 0.380957
Early stopping, best iteration is:
[199]	valid_0's multi_logloss: 0.38078
Fold 1 macro-F1: 0.68420

rich_temporal_magnitude_windows_peaks - Fold 2
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.366077
[200]	valid_0's multi_logloss: 0.342094
Early stopping, best iteration is:
[176]	valid_0's multi_logloss: 0.339102
Fold 2 macro-F1: 0.74746

rich_temporal_magnitude_windows_peaks - Fold 3
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.330636
[200]	valid_0's multi_logloss: 0.313125
Early stopping, best iteration is:
[166]	valid_0's multi_logloss: 0.309992
Fold 3 macro-F1: 0.74426

rich_temporal_magnitude_windows_peaks - Fold 4
Training 

Label
0    2815
1    3083
2      93
3     509
4      92
5     257
Name: count, dtype: int64

In [13]:
combination_summary = pd.DataFrame([
    {
        "experiment": r["experiment"],
        "n_features": r["n_features"],
        "mean_macro_f1": r["mean_macro_f1"],
        "std_macro_f1": r["std_macro_f1"],
        "oof_macro_f1": r["oof_macro_f1"],
        "submission_file": r["submission_file"]
    }
    for r in combination_results
])

baseline_score = combination_summary.loc[
    combination_summary["experiment"] == "baseline_raw_basic_stats",
    "mean_macro_f1"
].iloc[0]

combination_summary["gain_vs_baseline"] = combination_summary["mean_macro_f1"] - baseline_score

combination_summary = combination_summary.sort_values(
    "mean_macro_f1",
    ascending=False
).reset_index(drop=True)

display(combination_summary)

combination_summary.to_csv("combination_ablation_summary.csv", index=False)
print("Saved combination_ablation_summary.csv")


,experiment,n_features,mean_macro_f1,std_macro_f1,oof_macro_f1,submission_file,gain_vs_baseline
0,rich_temporal_magnitude_windows_peaks,686,0.719072,0.024223,0.728513,submission_combination_rich_temporal_magnitude...,0.043714
1,rich_temporal_magnitude,416,0.714133,0.031978,0.724798,submission_combination_rich_temporal_magnitude...,0.038775
2,rich_temporal_windows_peaks,426,0.708443,0.034491,0.717478,submission_combination_rich_temporal_windows_p...,0.033085
3,rich_temporal_no_magnitude,156,0.707243,0.036466,0.717671,submission_combination_rich_temporal_no_magnit...,0.031885
4,rich_temporal_fft,186,0.706670,0.038163,0.718170,submission_combination_rich_temporal_fft.csv,0.031312
5,rich_temporal_peaks,176,0.704889,0.030253,0.716331,submission_combination_rich_temporal_peaks.csv,0.029531
6,rich_temporal_windows,406,0.704652,0.032666,0.714942,submission_combination_rich_temporal_windows.csv,0.029294
7,baseline_raw_basic_stats,36,0.675358,0.034484,0.686435,submission_combination_baseline_raw_basic_stat...,0.000000


Saved combination_ablation_summary.csv
